In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed
from transformers.optimization import get_linear_schedule_with_warmup
from huggingface_hub import snapshot_download

base_model_name = "nghuyong/ernie-3.0-base-zh"
base_dir = "ernie-base-local"
output_dir = "ernie-domain-clf"
data_fpath = "wmt2425_en_zh_CN.jsonl"
labels = ["news", "social", "speech", "literary"]

max_tok_len = 256
seed = 42
batch_train_size = 16
batch_eval_size = 32
learning_rate = 2e-5
epochs = 5
warmup_ratio = 0.1

set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

local_model_path = snapshot_download(repo_id=base_model_name, local_dir=base_dir)
print(f"Model downloaded to: {local_model_path}")

with open(data_fpath, encoding="utf-8") as f:
    raw = [json.loads(line) for line in f]

label_to_id = {l: i for i, l in enumerate(labels)}
id_to_label = {i: l for l, i in label_to_id.items()}

random.shuffle(raw)
n = len(raw)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_raw = raw[:n_train]
val_raw = raw[n_train:n_train + n_val]
test_raw = raw[n_train + n_val:]

class ZhDomainDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, label_to_id):
        self.rows = rows
        self.tok = tokenizer
        self.max_len = max_len
        self.label_to_id = label_to_id
        self.text_field = 'target'
        self.label_field = 'domain'

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        ex = self.rows[i]
        enc = self.tok(
            str(ex[self.text_field]),
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.label_to_id[ex[self.label_field]])
        return enc

tokenizer = AutoTokenizer.from_pretrained(local_model_path, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    local_model_path,
    num_labels=len(labels),
    id2label=id_to_label,
    label2id=label_to_id,
).to(device)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds = ZhDomainDataset(train_raw, tokenizer, max_tok_len, label_to_id)
val_ds = ZhDomainDataset(val_raw, tokenizer, max_tok_len, label_to_id)
test_ds = ZhDomainDataset(test_raw, tokenizer, max_tok_len, label_to_id)

train_loader = DataLoader(train_ds, batch_size=batch_train_size, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=batch_eval_size, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=batch_eval_size, shuffle=False, collate_fn=collator)

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * max(1, epochs)
warmup_steps = int(warmup_ratio * total_steps)

if get_linear_schedule_with_warmup is not None and total_steps > 0:
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
else:
    scheduler = None

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

@torch.no_grad()
def evaluate(dataloader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
        logits = outputs.logits
        all_preds.append(logits.argmax(-1).detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "y_true": y_true,
        "y_pred": y_pred,
    }

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for step, batch in enumerate(progress, 1):
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        avg_loss = running_loss / step
        progress.set_postfix(loss=f"{avg_loss:.4f}")

    val_metrics = evaluate(val_loader)
    print(f"Epoch {epoch} | Val acc {val_metrics['accuracy']:.2f} | Val F1 {val_metrics['f1_macro']:.2f}")

test_metrics = evaluate(test_loader)
print(f"Test accuracy: {test_metrics['accuracy']}")
print(f"Test macro-F1: {test_metrics['f1_macro']}")

Path(output_dir).mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ds = load_dataset("haoranxu/ALMA-Human-Parallel", "zh-en")['train']

model_dir = "ernie-domain-clf"
out_dir   = "./domain_splits_train"
batch_size = 32

max_len = 256

SYSTEM_PROMPT = (
    "You are a translation assistant. Translate Chinese to English and output only the translation."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
model.eval()
id2label = model.config.id2label

zh_lines = [ln['translation']['zh'].strip() for ln in ds]
en_lines = [ln['translation']['en'].strip() for ln in ds]


all_preds = [None] * len(zh_lines)
for i in range(0, len(zh_lines), batch_size):
    batch_texts = zh_lines[i:i+batch_size]
    enc = tok(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1).tolist()


    for k, p in enumerate(preds):
        all_preds[i + k] = id2label[p]

assert all(x is not None for x in all_preds)

out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
buckets = {}

for idx, (zh, en, domain) in enumerate(zip(zh_lines, en_lines, all_preds)):
    rec = {
        "idx": idx,
        "domain": domain,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": zh},
            {"role": "assistant", "content": en},
        ],
    }
    buckets.setdefault(domain, []).append(rec)

for domain, recs in buckets.items():
    recs.sort(key=lambda r: r["idx"])
    out_path = out_dir / f"{domain}.jsonl"
    with out_path.open("w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(buckets)} domain files to {out_dir.resolve()}")
for d, recs in buckets.items():
    print(f" - {d}: {len(recs)}")